#Experiment 1
Model: YOLOv8n (Nano)
Image Size: 256×256
Epochs: 50
Batch Size: 16
Patience: Not Applied

In [ ]:
!pip install ultralytics pycocotools scikit-image tqdm opencv-python matplotlib --quiet

In [ ]:
import zipfile
import os

# Path to your zip file
zip_path = "/content/stage1_train.zip"   # change path if needed
extract_path = "/content/stage1_train"

# Create output folder if not exists
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)

✅ Unzipped to: /content/stage1_train


In [ ]:
# =========================================================
# YOLOv8 - Cell Nuclei Detection (DSB2018) using Labels CSV
# =========================================================

!pip install ultralytics opencv-python-headless tqdm --quiet

import os
import zipfile
import shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")


yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)


labels_df = pd.read_csv(csv_path)
print("✅ Loaded CSV:", labels_df.head())

def rle_decode(mask_rle, shape=(256, 256)):
    """Decode RLE to binary mask."""
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")


def mask_to_yolo_bbox(mask, img_w, img_h):
    """Extract bounding box from binary mask and convert to YOLO format."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:  # skip tiny noise
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        # Normalize to [0,1]
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        bw = w / img_w
        bh = h / img_h
        boxes.append([0, x_center, y_center, bw, bh])  # class=0
    return boxes

image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image to YOLO folder
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Get all masks for this image
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        all_boxes = []
        for rle in masks:
            if pd.isna(rle):
                continue
            mask = rle_decode(rle, (h, w))
            boxes = mask_to_yolo_bbox(mask, w, h)
            all_boxes.extend(boxes)

        # Write YOLO label file
        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for box in all_boxes:
                f.write(" ".join(map(str, box)) + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO format!")


yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")


model = YOLO("yolov8n.pt")  # small model for testing, use yolov8s/m/l for bigger
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=256,
    batch=16,
    device=0  # GPU
)


metrics = model.val()
print("✅ Validation metrics:", metrics)


!mkdir -p /content/yolo_results
results = model.predict(source=os.path.join(yolo_base, "images", "val"), save=True, project="/content/yolo_results")
print("✅ Inference complete! Results saved in /content/yolo_results")


✅ Unzipped stage1_train.zip successfully!
✅ Loaded CSV:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:03<00:00, 40.58it/s]


✅ Dataset prepared in YOLO format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

In [ ]:
!pip install medpy --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 19.5 MB/s eta 0:00:00


In [ ]:
# =========================================================
# Evaluate YOLOv8 model on validation set using Dice Score
# =========================================================
from medpy.metric import binary
import numpy as np
import cv2
import os
from tqdm import tqdm

# 1️⃣ Load trained model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # update path if different

# 2️⃣ Helper functions
def bbox_to_mask(boxes, h, w):
    """Convert YOLO-format boxes to binary mask"""
    mask = np.zeros((h, w), dtype=np.uint8)
    for box in boxes:
        cls, x_c, y_c, bw, bh = box
        x1 = int((x_c - bw/2) * w)
        y1 = int((y_c - bh/2) * h)
        x2 = int((x_c + bw/2) * w)
        y2 = int((y_c + bh/2) * h)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w-1, x2), min(h-1, y2)
        mask[y1:y2, x1:x2] = 1
    return mask

def rle_decode(mask_rle, shape):
    """Decode RLE string into binary mask"""
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# 3️⃣ Evaluate Dice
dice_scores = []

for img_id in tqdm(val_ids, desc="Dice Eval"):
    img_dir = os.path.join(extract_dir, img_id, "images")
    img_files = os.listdir(img_dir)
    if not img_files:
        continue
    img_path = os.path.join(img_dir, img_files[0])
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # --- Ground truth mask ---
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        if pd.isna(rle):
            continue
        gt_mask |= rle_decode(rle, (h, w))

    # --- YOLO prediction (from trained weights) ---
    results = model.predict(img_path, imgsz=256, conf=0.25, verbose=False)
    pred_boxes = results[0].boxes.xywhn.cpu().numpy()  # [x_c, y_c, w, h]
    pred_classes = results[0].boxes.cls.cpu().numpy()
    pred_all = [[int(c), *b] for c, b in zip(pred_classes, pred_boxes)]

    pred_mask = bbox_to_mask(pred_all, h, w)

    # --- Dice computation ---
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

Dice Eval: 100%|██████████| 134/134 [00:05<00:00, 26.61it/s]

✅ Mean Dice Score on Validation Set: 0.7282


#Experiment 2
Model: YOLOv8n (Nano)
Image Size: 256×256
Epochs: 100
Batch Size: 16
Patience: 15

In [ ]:
# =========================================================
# YOLOv8 for Cell Nuclei Detection with Dice Evaluation
# Optimized for Colab: Faster Training + Augmentations
# =========================================================

!pip install ultralytics -q
from ultralytics import YOLO
import os, cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =========================================================
# Step 1: Prepare Dataset
# =========================================================
!unzip -q /content/stage1_train.zip -d /content/

data_dir = "/content/stage1_train"
labels_csv = "/content/stage1_train_labels.csv"

df = pd.read_csv(labels_csv)
df['filename'] = df['ImageId'].astype(str) + ".png"

# Train-val split (80-20)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

os.makedirs("/content/yolo_cell_nuclei/images/train", exist_ok=True)
os.makedirs("/content/yolo_cell_nuclei/images/val", exist_ok=True)
os.makedirs("/content/yolo_cell_nuclei/labels/train", exist_ok=True)
os.makedirs("/content/yolo_cell_nuclei/labels/val", exist_ok=True)

def create_yolo_labels(df, split):
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_id = row['ImageId']
        img_path = os.path.join(data_dir, img_id, "images", img_id + ".png")
        mask_dir = os.path.join(data_dir, img_id, "masks")

        # Copy image
        img_out = f"/content/yolo_cell_nuclei/images/{split}/{img_id}.png"
        if not os.path.exists(img_out):
            !cp "{img_path}" "{img_out}"

        # Convert masks to bounding boxes
        h, w = cv2.imread(img_path, 0).shape
        bboxes = []
        for mfile in os.listdir(mask_dir):
            mask = cv2.imread(os.path.join(mask_dir, mfile), 0)
            if mask is None: continue
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for c in contours:
                x, y, bw, bh = cv2.boundingRect(c)
                # Normalize for YOLO
                cx = (x + bw/2) / w
                cy = (y + bh/2) / h
                bw /= w
                bh /= h
                bboxes.append([0, cx, cy, bw, bh])

        # Save label file
        with open(f"/content/yolo_cell_nuclei/labels/{split}/{img_id}.txt", "w") as f:
            for bb in bboxes:
                f.write(" ".join(map(str, bb)) + "\n")

create_yolo_labels(train_df, "train")
create_yolo_labels(val_df, "val")

# =========================================================
# Step 2: YAML Dataset Config
# =========================================================
dataset_yaml = """
path: /content/yolo_cell_nuclei
train: images/train
val: images/val

nc: 1
names: ['nucleus']
"""
with open("/content/cell_nuclei.yaml", "w") as f:
    f.write(dataset_yaml)

# =========================================================
# Step 3: Train YOLOv8
# =========================================================
model = YOLO("yolov8n.pt")  # use nano model for speed

model.train(
    data="/content/cell_nuclei.yaml",
    epochs=100,
    patience=15,
    batch=16,
    imgsz=256,
    cache="ram",      # cache dataset in RAM
    augment=True,     # YOLO built-in augmentations
    mosaic=1.0,       # enable mosaic
    mixup=0.2,        # mixup augmentation
    degrees=10,
    translate=0.1,
    scale=0.1,
    shear=2,
    flipud=0.5,
    fliplr=0.5
)

# =========================================================
# Step 4: Evaluate Dice Score
# =========================================================
from glob import glob

def dice_score(mask1, mask2):
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)
    inter = np.logical_and(mask1, mask2).sum()
    return (2. * inter) / (mask1.sum() + mask2.sum() + 1e-8)

# Load best weights
best_model = YOLO("/content/runs/detect/train/weights/best.pt")

val_imgs = glob("/content/yolo_cell_nuclei/images/val/*.png")
dice_scores = []

for img_path in tqdm(val_imgs):
    # Predict bounding boxes
    results = best_model(img_path, imgsz=256, conf=0.25)[0]

    # GT mask
    img_id = os.path.basename(img_path).replace(".png", "")
    mask_dir = os.path.join(data_dir, img_id, "masks")
    gt_mask = np.zeros(cv2.imread(img_path, 0).shape, dtype=np.uint8)
    for mfile in os.listdir(mask_dir):
        m = cv2.imread(os.path.join(mask_dir, mfile), 0)
        gt_mask = np.logical_or(gt_mask, m).astype(np.uint8)

    # Pred mask from boxes
    pred_mask = np.zeros_like(gt_mask)
    for box in results.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        pred_mask[y1:y2, x1:x2] = 1

    dice_scores.append(dice_score(gt_mask, pred_mask))

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")


100%|██████████| 5893/5893 [09:46<00:00, 10.05it/s]


Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cell_nuclei.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=True, pose=12.0, pretr

  0%|          | 0/651 [00:00<?, ?it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/aaa52100fafaa50877e777229cdf6cde7c422f145ff6719449b80631d9a3b0f6.png: 192x256 139 nucleuss, 33.9ms
Speed: 0.8ms preprocess, 33.9ms inference, 1.8ms postprocess per image at shape (1, 3, 192, 256)


  0%|          | 1/651 [00:00<02:45,  3.93it/s]

...


image 1/1 /content/yolo_cell_nuclei/images/val/b1e3aeb0c56261c17eb71c747d116057b8da7e8c8a6845bdc01b2b3ee2299229.png: 256x256 18 nucleuss, 5.4ms
Speed: 0.4ms preprocess, 5.4ms inference, 1.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/0acd2c223d300ea55d0546797713851e818e5c697d073b7f4091b96ce0f3d2fe.png: 256x256 8 nucleuss, 5.6ms
Speed: 0.4ms preprocess, 5.6ms inference, 1.0ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/4dbbb275960ab9e4ec2c66c8d3000f7c70c8dce5112df591b95db84e25efa6e9.png: 192x256 175 nucleuss, 6.0ms
Speed: 0.6ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 192, 256)


100%|█████████▉| 648/651 [00:31<00:00, 21.48it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/953211bcc0192e2298087d30e708dba68def9e0c13a3ff3326a18b0962c63adc.png: 224x256 15 nucleuss, 6.3ms
Speed: 0.6ms preprocess, 6.3ms inference, 1.3ms postprocess per image at shape (1, 3, 224, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/af8621ef0db8c26b0bce6385bd5609b584bfd678fcf7a234b8a15e6bb05c15ac.png: 256x256 22 nucleuss, 5.8ms
Speed: 0.4ms preprocess, 5.8ms inference, 1.2ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/4d09672bcf5a2661eea00891bbb8191225a06619a849aece37ad10d9dedbde3e.png: 192x256 30 nucleuss, 5.8ms
Speed: 0.6ms preprocess, 5.8ms inference, 1.1ms postprocess per image at shape (1, 3, 192, 256)


100%|██████████| 651/651 [00:31<00:00, 20.66it/s]

✅ Mean Dice Score on Validation Set: 0.7921


#Experiment 3
Model: YOLOv8s-seg
Image Size: 256×256
Epochs: 100
Batch Size: 16
Patience: 20

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary


zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")


yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

def rle_decode(mask_rle, shape):
    """Decode RLE into binary mask"""
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

def mask_to_yolo_polygons(mask, img_w, img_h):
    """Convert binary mask to YOLO polygon format"""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:  # skip tiny noise
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])  # normalize
        if len(poly) >= 6:  # at least 3 points
            polys.append(poly)
    return polys

image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg format!")

yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")


model = YOLO("yolov8s-seg.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=256,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=1e-3,
    patience=20,
    # --- On-the-fly augmentations ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3
)


metrics = model.val()
print("✅ Validation metrics:", metrics)

!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

dice_scores = []

for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 47.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

               

Processing val: 100%|██████████| 134/134 [00:03<00:00, 34.80it/s]


✅ Dataset prepared in YOLO-Seg format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimi

#Experiment 4
Model: YOLOv8m-seg
Image Size: 512×512
Epochs: 250
Batch Size: 8
Patience: 50

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=250,          # maximum epochs
    imgsz=512,           # higher resolution
    batch=8,             # adjust based on GPU memory
    device=0,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    patience=50,         # early stopping
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:03<00:00, 36.54it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=250, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False

In [ ]:
import pandas as pd

csv_path = "/content/stage1_train_labels.csv"
df = pd.read_csv(csv_path)

print("✅ Columns in CSV:", df.columns.tolist())
print("✅ First few rows:\n", df.head())


✅ Columns in CSV: ['ImageId', 'EncodedPixels']
✅ First few rows:
                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


#Experiment 5
Model: YOLOv8m-seg
Image Size: 512×512
Epochs: 100
Batch Size: 16
Patience: 10

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=100,          # maximum epochs
    imgsz=512,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    patience=10,         # early stopping
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 44.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

               

Processing val: 100%|██████████| 134/134 [00:03<00:00, 39.61it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False

#Experiment 6
Model: YOLOv8m-seg
Image Size: 512×512
Epochs: 100
Batch Size: 16
Patience: 10

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=100,          # maximum epochs
    imgsz=512,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    patience=10,         # early stopping
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:03<00:00, 38.02it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=Fals

#Experiment 7
Model: YOLOv8m-seg
Image Size: 512×512
Epochs: 25
Batch Size: 16
Patience: 10

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=25,          # maximum epochs
    imgsz=512,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    optimizer="AdamW",
    lr0=1e-4,            # early stopping
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:03<00:00, 37.12it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False

#Experiment 8
Model: YOLOv8m-seg
Image Size: 640×640
Epochs: 25
Batch Size: 16
Patience: Not Applied

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=25,          # maximum epochs
    imgsz=640,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    # patience = 10,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:03<00:00, 37.77it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False

#Experiment 9
Model: YOLOv8m-seg
Image Size: 640×640
Epochs: 50
Batch Size: 16
Patience: Not Applied

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=50,          # maximum epochs
    imgsz=640,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    # patience = 10,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 45.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

               

Processing val: 100%|██████████| 134/134 [00:03<00:00, 39.53it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False,

#Experiment 10
Model: YOLOv8m-seg
Image Size: 640×640
Epochs: 100
Batch Size: 16
Patience: 10

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=100,          # maximum epochs
    imgsz=640,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    patience = 10,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

                                       EncodedPixels  
0  6908 1 7161 8 7417 8 7672 9 7928 9 8184 9 8440...  
1  36269 7 36523 11 36778 13 37033 15 37288 17 37...  
2  19919 6 20174 8 20429 10 20685 11 20941 12 211...  
3  18671 6 18926 8 19181 9 19436 10 19691 11 1994...  
4            40158 3 40413 5 40669 5 40925 5 41182 3  


Processing val: 100%|██████████| 134/134 [00:04<00:00, 33.02it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False

#Experiment 11
Model: YOLOv8m-seg
Image Size: 640×640
Epochs: 150
Batch Size: 16
Patience: Not Applied

In [ ]:
!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# ==========================
# 1️⃣ UNZIP DATASET
# ==========================
zip_path = "/content/stage1_train.zip"
csv_path = "/content/stage1_train_labels.csv"
extract_dir = "/content/stage1_train"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Unzipped stage1_train.zip successfully!")

# ==========================
# 2️⃣ PREPARE YOLO FOLDERS
# ==========================
yolo_base = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split), exist_ok=True)

# ==========================
# 3️⃣ LOAD CSV LABELS
# ==========================
labels_df = pd.read_csv(csv_path)
print("✅ Loaded labels:", labels_df.head())

# ==========================
# 4️⃣ RLE DECODE
# ==========================
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# ==========================
# 5️⃣ POLYGON CONVERSION
# ==========================
def mask_to_yolo_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# ==========================
# 6️⃣ SPLIT DATA
# ==========================
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# ==========================
# 7️⃣ PROCESS SPLIT
# ==========================
def process_split(ids, split):
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(extract_dir, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Save image
        dst_img_path = os.path.join(yolo_base, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Create YOLO-Seg label (polygon format)
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_yolo_polygons(mask, w, h)
            polygons.extend(polys)

        label_path = os.path.join(yolo_base, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))  # class 0
                f.write(line + "\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO-Seg polygon format!")

# ==========================
# 8️⃣ CREATE YAML
# ==========================
yaml_path = os.path.join(yolo_base, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {yolo_base}
train: images/train
val: images/val
names:
  0: nucleus
""")
print("✅ Created dataset.yaml")

# ==========================
# 9️⃣ TRAIN YOLOv8-Seg
# ==========================
model = YOLO("yolov8m-seg.pt")  # Medium model for better features

model.train(
    data=yaml_path,
    epochs=150,          # maximum epochs
    imgsz=640,           # higher resolution
    batch=16,             # adjust based on GPU memory
    device=0,
    # patience = 10,
    optimizer="AdamW",
    lr0=1e-4,            # smaller LR for stable training
    # --- moderate augmentations ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5, translate=0.05, scale=0.1, shear=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.1
)

# ==========================
# 10️⃣ VALIDATION METRICS
# ==========================
metrics = model.val()
print("✅ Validation metrics:", metrics)

# ==========================
# 11️⃣ INFERENCE
# ==========================
!mkdir -p /content/yolo_results_seg
results = model.predict(
    source=os.path.join(yolo_base, "images", "val"),
    save=True,
    project="/content/yolo_results_seg"
)
print("✅ Inference complete! Results in /content/yolo_results_seg")

# ==========================
# 12️⃣ COMPUTE DICE SCORE
# ==========================
dice_scores = []
for r in results:
    img_path = r.path
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None:
        for m in r.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    # Post-processing
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = (pred_mask > 0).astype(np.uint8)

    # Dice
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 48.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Unzipped stage1_train.zip successfully!
✅ Loaded labels:                                              ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

               

Processing val: 100%|██████████| 134/134 [00:03<00:00, 37.85it/s]


✅ Dataset prepared in YOLO-Seg polygon format!
✅ Created dataset.yaml
Ultralytics 8.3.198 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False

#Experiment 12
Model: YOLOv8l-seg
Image Size: 640×640
Epochs: 100
Batch Size: 16
Patience: Not Applied

In [ ]:
# =========================================================
# YOLOv8-Seg for Cell Nuclei Segmentation (No Empty Labels)
# =========================================================

!pip install ultralytics opencv-python-headless tqdm medpy --quiet

import os, zipfile, shutil, cv2, numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary

# -----------------------------
# 1️⃣ Unzip Dataset
# -----------------------------
ZIP_PATH = "/content/stage1_train.zip"
EXTRACT_DIR = "/content/stage1_train"
CSV_PATH = "/content/stage1_train_labels.csv"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print("✅ Dataset unzipped successfully!")

# -----------------------------
# 2️⃣ Load Labels CSV
# -----------------------------
labels_df = pd.read_csv(CSV_PATH)
labels_df = labels_df.dropna(subset=["EncodedPixels"])  # remove images with no masks
print("✅ Loaded labels CSV. Sample:")
print(labels_df.head())

# -----------------------------
# 3️⃣ Split Data (Train/Val)
# -----------------------------
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# -----------------------------
# 4️⃣ Create YOLOv8 Folders
# -----------------------------
YOLO_BASE = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(YOLO_BASE, "images", split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_BASE, "labels", split), exist_ok=True)

# -----------------------------
# 5️⃣ RLE Decode Function
# -----------------------------
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# -----------------------------
# 6️⃣ Convert Mask -> YOLO Polygon
# -----------------------------
def mask_to_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 5:  # include small masks
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x / img_w, y / img_h])
        if len(poly) >= 6:  # at least 3 points
            polys.append(poly)
    return polys

# -----------------------------
# 7️⃣ Process Train/Val Splits
# -----------------------------
def process_split(ids, split):
    skipped = 0
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(EXTRACT_DIR, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            skipped += 1
            continue
        img_path = os.path.join(img_dir, img_files[0])

        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get masks for this image
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_polygons(mask, w, h)
            polygons.extend(polys)

        if len(polygons) == 0:
            skipped += 1
            continue  # skip images with no valid polygons

        # Save image
        dst_img_path = os.path.join(YOLO_BASE, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img_path)

        # Save label file
        label_path = os.path.join(YOLO_BASE, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                line = "0 " + " ".join(map(str, poly))
                f.write(line + "\n")
    print(f"Skipped {skipped} images in {split} (no masks)")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ Dataset prepared in YOLO polygon format (no empty labels)")

# -----------------------------
# 8️⃣ Create YAML Config
# -----------------------------
yaml_path = os.path.join(YOLO_BASE, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {YOLO_BASE}
train: images/train
val: images/val
nc: 1
names:
  0: nucleus
""")
print("✅ YOLO dataset YAML created!")

# -----------------------------
# 9️⃣ Train YOLOv8-Seg
# -----------------------------
model = YOLO("yolov8l-seg.pt")  # use yolov8n-seg / m-seg for smaller/faster

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=2e-4,
    weight_decay=5e-4,
    cos_lr=True,
    warmup_epochs=5,
    # patience=30,
    augment=True,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.7,
    degrees=10, translate=0.1, scale=0.2, shear=5,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.2, copy_paste=0.3
)

# -----------------------------
# 10️⃣ Validation & Dice
# -----------------------------
best_model = YOLO("/content/runs/segment/train/weights/best.pt")
val_imgs = [f for f in os.listdir(os.path.join(YOLO_BASE,"images","val")) if f.endswith(".png")]
dice_scores = []

for img_name in tqdm(val_imgs, desc="Calculating Dice"):
    img_id = os.path.splitext(img_name)[0]
    img_path = os.path.join(YOLO_BASE,"images","val",img_name)
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Ground truth
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle, (h, w))

    # Prediction
    results = best_model(img_path, imgsz=256, conf=0.25)[0]
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if results.masks is not None:
        for m in results.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m, (w, h))
            pred_mask |= m

    pred_mask = (pred_mask > 0).astype(np.uint8)
    if gt_mask.sum() == 0 and pred_mask.sum() == 0:
        dice = 1.0
    else:
        dice = binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 36.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset unzipped successfully!
✅ Loaded labels CSV. Sample:
                                             ImageId  \
0  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
1  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
2  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
3  00071198d059ba7f5914a526d124d28e6d010c92466da2...   
4  00071198d059ba7f5914a526d124d28e6d010c92466da2...   

            

Processing train: 100%|██████████| 536/536 [00:13<00:00, 39.03it/s]


Skipped 0 images in train (no masks)


Processing val: 100%|██████████| 134/134 [00:03<00:00, 35.59it/s]


Skipped 0 images in val (no masks)
✅ Dataset prepared in YOLO polygon format (no empty labels)
✅ YOLO dataset YAML created!
Ultralytics 8.3.199 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.7, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8l-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name

...


image 1/1 /content/yolo_cell_nuclei/images/val/0b2e702f90aee4fff2bc6e4326308d50cf04701082e718d4f831c8959fbcda93.png: 256x256 5 nucleuss, 11.6ms
Speed: 0.5ms preprocess, 11.6ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/d1dbc6ee7c44a7027e935d040e496793186b884a1028d0e26284a206c6f5aff0.png: 256x256 11 nucleuss, 11.8ms
Speed: 0.4ms preprocess, 11.8ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/3a3fee427e6ef7dfd0d82681e2bcee2d054f80287aea7dfa3fa4447666f929b9.png: 256x256 65 nucleuss, 11.9ms
Speed: 0.6ms preprocess, 11.9ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/fadeb0ab092833f27daaeb3e24223eb090f9536b83f68cde8f49df7c544f711b.png: 256x256 41 nucleuss, 11.7ms
Speed: 0.5ms preprocess, 11.7ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)


Calculating Dice:  99%|█████████▊| 132/134 [00:08<00:00, 20.70it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/29dd28df98ee51b4ab1a87f5509538ecc3e4697fc57c40c6165658f61b0d8e3a.png: 256x256 56 nucleuss, 11.7ms
Speed: 0.5ms preprocess, 11.7ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/62570c4ff1c5ab6d9d383aba9f25e604768520b4266afd40fdf4734a694c8bc3.png: 256x256 12 nucleuss, 12.2ms
Speed: 0.4ms preprocess, 12.2ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)


Calculating Dice: 100%|██████████| 134/134 [00:08<00:00, 16.19it/s]

✅ Mean Dice Score on Validation Set: 0.7371


#Experiment 13
Model: YOLOv8x-seg
Image Size: 768×768
Epochs: 150
Batch Size: 8
Patience: 30

In [ ]:
# =========================================================
# Optimized YOLOv8-Seg Pipeline for Cell Nuclei Segmentation
# =========================================================

!pip install ultralytics opencv-python-headless tqdm medpy albumentations --quiet

import os, zipfile, shutil, cv2, numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary
import albumentations as A

# -----------------------------
# 1️⃣ Unzip Dataset
# -----------------------------
ZIP_PATH = "/content/stage1_train.zip"
EXTRACT_DIR = "/content/stage1_train"
CSV_PATH = "/content/stage1_train_labels.csv"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print("✅ Dataset unzipped!")

# -----------------------------
# 2️⃣ Load Labels
# -----------------------------
labels_df = pd.read_csv(CSV_PATH)
labels_df = labels_df.dropna(subset=["EncodedPixels"])  # skip empty masks
print("✅ Loaded labels CSV:", labels_df.shape)

# -----------------------------
# 3️⃣ Split Train/Val
# -----------------------------
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# -----------------------------
# 4️⃣ YOLO Folder Structure
# -----------------------------
YOLO_BASE = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(YOLO_BASE, "images", split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_BASE, "labels", split), exist_ok=True)

# -----------------------------
# 5️⃣ RLE Decode
# -----------------------------
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# -----------------------------
# 6️⃣ Mask -> Polygon
# -----------------------------
def mask_to_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 3:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x/img_w, y/img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# -----------------------------
# 7️⃣ Process Splits
# -----------------------------
def process_split(ids, split):
    skipped = 0
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(EXTRACT_DIR, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            skipped += 1
            continue
        img_path = os.path.join(img_dir, img_files[0])
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # get masks
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_polygons(mask, w, h)
            polygons.extend(polys)
        if len(polygons) == 0:
            skipped += 1
            continue

        # save image
        dst_img = os.path.join(YOLO_BASE, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img)

        # save label
        label_path = os.path.join(YOLO_BASE, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                f.write("0 " + " ".join(map(str, poly)) + "\n")
    print(f"Skipped {skipped} images in {split} (no valid masks)")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ YOLOv8 polygon dataset ready!")

# -----------------------------
# 8️⃣ YAML Config
# -----------------------------
yaml_path = os.path.join(YOLO_BASE, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {YOLO_BASE}
train: images/train
val: images/val
nc: 1
names:
  0: nucleus
""")
print("✅ YOLO dataset YAML created!")

# -----------------------------
# 9️⃣ Train YOLOv8-Seg
# -----------------------------
model = YOLO("yolov8x-seg.pt")  # use x-seg for best accuracy

model.train(
    data=yaml_path,
    epochs=150,
    imgsz=768,
    batch=8,  # adjust if OOM
    device=0,
    optimizer="AdamW",
    lr0=2e-4,
    weight_decay=5e-4,
    cos_lr=True,
    warmup_epochs=5,
    patience=30,
    augment=True,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.7,
    degrees=15, translate=0.1, scale=0.2, shear=5,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.2, copy_paste=0.3
)


✅ Dataset unzipped!
✅ Loaded labels CSV: (29461, 2)


Processing train: 100%|██████████| 536/536 [00:14<00:00, 36.95it/s]


Skipped 0 images in train (no valid masks)


Processing val: 100%|██████████| 134/134 [00:04<00:00, 33.03it/s]


Skipped 0 images in val (no valid masks)
✅ YOLOv8 polygon dataset ready!
✅ YOLO dataset YAML created!
Ultralytics 8.3.199 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.7, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=Fa

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ea1e808e900>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [ ]:
 #🔟 Validation Dice Score
# -----------------------------
best_model = YOLO("/content/runs/segment/train2/weights/best.pt")
val_imgs = [f for f in os.listdir(os.path.join(YOLO_BASE,"images","val")) if f.endswith(".png")]
dice_scores = []

for img_name in tqdm(val_imgs, desc="Dice Computation"):
    img_id = os.path.splitext(img_name)[0]
    img_path = os.path.join(YOLO_BASE,"images","val",img_name)
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # GT
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h,w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle,(h,w))

    # Prediction
    results = best_model(img_path, imgsz=256, conf=0.25)[0]
    pred_mask = np.zeros((h,w), dtype=np.uint8)
    if results.masks is not None:
        for m in results.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m,(w,h))
            pred_mask |= m
    pred_mask = (pred_mask>0).astype(np.uint8)

    # Morphological clean
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)

    # Dice
    dice = 1.0 if gt_mask.sum()==0 and pred_mask.sum()==0 else binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")


Dice Computation:   0%|          | 0/134 [00:00<?, ?it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/1d02c4b5921e916b9ddfb2f741fd6cf8d0e571ad51eb20e021c826b5fb87350e.png: 224x256 12 nucleuss, 77.6ms
Speed: 0.8ms preprocess, 77.6ms inference, 2.8ms postprocess per image at shape (1, 3, 224, 256)


Dice Computation:   1%|          | 1/134 [00:00<01:49,  1.22it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/30f65741053db713b3f328d31d3234b6fedbe31df65c1a8ea29be28146cab789.png: 256x256 26 nucleuss, 13.0ms
Speed: 0.6ms preprocess, 13.0ms inference, 2.2ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/4e1c889de3764694d0dea41e5682fedb265eaf2cdbe72ff6c1f518747d709464.png: 192x256 35 nucleuss, 77.8ms
Speed: 0.8ms preprocess, 77.8ms inference, 2.2ms postprocess per image at shape (1, 3, 192, 256)


Dice Computation:   2%|▏         | 3/134 [00:00<00:36,  3.61it/s]

...


image 1/1 /content/yolo_cell_nuclei/images/val/0b2e702f90aee4fff2bc6e4326308d50cf04701082e718d4f831c8959fbcda93.png: 256x256 5 nucleuss, 12.3ms
Speed: 0.5ms preprocess, 12.3ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/d1dbc6ee7c44a7027e935d040e496793186b884a1028d0e26284a206c6f5aff0.png: 256x256 10 nucleuss, 12.3ms
Speed: 0.5ms preprocess, 12.3ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/3a3fee427e6ef7dfd0d82681e2bcee2d054f80287aea7dfa3fa4447666f929b9.png: 256x256 57 nucleuss, 11.9ms
Speed: 0.4ms preprocess, 11.9ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/fadeb0ab092833f27daaeb3e24223eb090f9536b83f68cde8f49df7c544f711b.png: 256x256 45 nucleuss, 12.2ms
Speed: 0.5ms preprocess, 12.2ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)


Dice Computation:  99%|█████████▊| 132/134 [00:08<00:00, 20.82it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/29dd28df98ee51b4ab1a87f5509538ecc3e4697fc57c40c6165658f61b0d8e3a.png: 256x256 55 nucleuss, 12.5ms
Speed: 0.5ms preprocess, 12.5ms inference, 2.2ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/62570c4ff1c5ab6d9d383aba9f25e604768520b4266afd40fdf4734a694c8bc3.png: 256x256 10 nucleuss, 12.6ms
Speed: 0.5ms preprocess, 12.6ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)


Dice Computation: 100%|██████████| 134/134 [00:08<00:00, 15.91it/s]

✅ Mean Dice Score on Validation Set: 0.7360


#Experiment 14
Model: YOLOv8x-seg
Image Size: 768×768
Epochs: 150
Batch Size: 8
Patience: 30

In [ ]:
# =========================================================
# Advanced YOLOv8-Seg Pipeline for Cell Nuclei Segmentation
# =========================================================

!pip install ultralytics opencv-python-headless tqdm medpy albumentations --quiet

import os, zipfile, shutil, cv2, numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from medpy.metric import binary
import albumentations as A

# -----------------------------
# 1️⃣ Unzip Dataset
# -----------------------------
ZIP_PATH = "/content/stage1_train.zip"
EXTRACT_DIR = "/content/stage1_train"
CSV_PATH = "/content/stage1_train_labels.csv"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print("✅ Dataset unzipped!")

# -----------------------------
# 2️⃣ Load Labels
# -----------------------------
labels_df = pd.read_csv(CSV_PATH)
labels_df = labels_df.dropna(subset=["EncodedPixels"])  # skip empty masks
print("✅ Loaded labels CSV:", labels_df.shape)

# -----------------------------
# 3️⃣ Split Train/Val
# -----------------------------
image_ids = labels_df["ImageId"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

# -----------------------------
# 4️⃣ YOLO Folder Structure
# -----------------------------
YOLO_BASE = "/content/yolo_cell_nuclei"
for split in ["train", "val"]:
    os.makedirs(os.path.join(YOLO_BASE, "images", split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_BASE, "labels", split), exist_ok=True)

# -----------------------------
# 5️⃣ RLE Decode
# -----------------------------
def rle_decode(mask_rle, shape):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)
    s = np.array(mask_rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

# -----------------------------
# 6️⃣ Mask -> Polygon
# -----------------------------
def mask_to_polygons(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 3:
            continue
        poly = []
        for p in cnt:
            x, y = p[0]
            poly.extend([x/img_w, y/img_h])
        if len(poly) >= 6:
            polys.append(poly)
    return polys

# -----------------------------
# 7️⃣ Process Splits
# -----------------------------
def process_split(ids, split):
    skipped = 0
    for img_id in tqdm(ids, desc=f"Processing {split}"):
        img_dir = os.path.join(EXTRACT_DIR, img_id, "images")
        img_files = os.listdir(img_dir)
        if not img_files:
            skipped += 1
            continue
        img_path = os.path.join(img_dir, img_files[0])
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # get masks
        masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
        polygons = []
        for rle in masks:
            mask = rle_decode(rle, (h, w))
            polys = mask_to_polygons(mask, w, h)
            polygons.extend(polys)
        if len(polygons) == 0:
            skipped += 1
            continue

        # save image
        dst_img = os.path.join(YOLO_BASE, "images", split, f"{img_id}.png")
        shutil.copy(img_path, dst_img)

        # save label
        label_path = os.path.join(YOLO_BASE, "labels", split, f"{img_id}.txt")
        with open(label_path, "w") as f:
            for poly in polygons:
                f.write("0 " + " ".join(map(str, poly)) + "\n")
    print(f"Skipped {skipped} images in {split} (no valid masks)")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("✅ YOLOv8 polygon dataset ready!")

# -----------------------------
# 8️⃣ YAML Config
# -----------------------------
yaml_path = os.path.join(YOLO_BASE, "cell_nuclei.yaml")
with open(yaml_path, "w") as f:
    f.write(f"""
path: {YOLO_BASE}
train: images/train
val: images/val
nc: 1
names:
  0: nucleus
""")
print("✅ YOLO dataset YAML created!")

# -----------------------------
# 9️⃣ Advanced Augmentation Pipeline
# -----------------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.2)
])

# -----------------------------
# 🔟 Train YOLOv8-Seg
# -----------------------------
model = YOLO("yolov8x-seg.pt")  # extra-large for accuracy

model.train(
    data=yaml_path,
    epochs=150,
    imgsz=768,
    batch=8,  # adjust if OOM
    device=0,
    optimizer="AdamW",
    lr0=2e-4,
    weight_decay=5e-4,
    cos_lr=True,
    warmup_epochs=5,
    patience=30,
    augment=True,
    mosaic=1.0, mixup=0.2, copy_paste=0.3
)

# -----------------------------
# 11️⃣ Validation Dice Score with Post-Processing
# -----------------------------
best_model = YOLO("/content/runs/segment/train/weights/best.pt")
val_imgs = [f for f in os.listdir(os.path.join(YOLO_BASE,"images","val")) if f.endswith(".png")]
dice_scores = []

def post_process(mask):
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    # remove small components
    num_labels, labels_im = cv2.connectedComponents(mask)
    final = np.zeros_like(mask)
    for i in range(1, num_labels):
        if (labels_im==i).sum() > 10:  # min size filter
            final[labels_im==i] = 1
    return final

for img_name in tqdm(val_imgs, desc="Dice Computation"):
    img_id = os.path.splitext(img_name)[0]
    img_path = os.path.join(YOLO_BASE,"images","val",img_name)
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # GT
    masks = labels_df[labels_df["ImageId"] == img_id]["EncodedPixels"]
    gt_mask = np.zeros((h,w), dtype=np.uint8)
    for rle in masks:
        gt_mask |= rle_decode(rle,(h,w))

    # Prediction
    results = best_model(img_path, imgsz=256, conf=0.25)[0]
    pred_mask = np.zeros((h,w), dtype=np.uint8)
    if results.masks is not None:
        for m in results.masks.data:
            m = m.cpu().numpy().astype(np.uint8)
            m = cv2.resize(m,(w,h))
            pred_mask |= m
    pred_mask = (pred_mask>0).astype(np.uint8)
    pred_mask = post_process(pred_mask)

    # Dice
    dice = 1.0 if gt_mask.sum()==0 and pred_mask.sum()==0 else binary.dc(pred_mask, gt_mask)
    dice_scores.append(dice)

print(f"✅ Mean Dice Score on Validation Set: {np.mean(dice_scores):.4f}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 13.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 47.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset unzipped!
✅ Loaded labels CSV: (29461, 2)


Processing train: 100%|██████████| 536/536 [00:13<00:00, 39.47it/s]


Skipped 0 images in train (no valid masks)


Processing val: 100%|██████████| 134/134 [00:03<00:00, 36.45it/s]
/tmp/ipython-input-294040379.py:144: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),


Skipped 0 images in val (no valid masks)
✅ YOLOv8 polygon dataset ready!
✅ YOLO dataset YAML created!
Ultralytics 8.3.200 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_cell_nuclei/cell_nuclei.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=F

...


image 1/1 /content/yolo_cell_nuclei/images/val/e9b8ad127f2163438b6236c74938f43d7b4863aaf39a16367f4af59bfd96597b.png: 256x256 11 nucleuss, 12.8ms
Speed: 0.5ms preprocess, 12.8ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/0b2e702f90aee4fff2bc6e4326308d50cf04701082e718d4f831c8959fbcda93.png: 256x256 5 nucleuss, 12.2ms
Speed: 0.4ms preprocess, 12.2ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/d1dbc6ee7c44a7027e935d040e496793186b884a1028d0e26284a206c6f5aff0.png: 256x256 9 nucleuss, 12.4ms
Speed: 0.4ms preprocess, 12.4ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/3a3fee427e6ef7dfd0d82681e2bcee2d054f80287aea7dfa3fa4447666f929b9.png: 256x256 59 nucleuss, 12.5ms
Speed: 0.5ms preprocess, 12.5ms inference, 2.2ms postprocess per image at shape (1, 3, 256, 256)


Dice Computation:  98%|█████████▊| 131/134 [00:09<00:00, 19.26it/s]


image 1/1 /content/yolo_cell_nuclei/images/val/fadeb0ab092833f27daaeb3e24223eb090f9536b83f68cde8f49df7c544f711b.png: 256x256 43 nucleuss, 12.4ms
Speed: 0.6ms preprocess, 12.4ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/29dd28df98ee51b4ab1a87f5509538ecc3e4697fc57c40c6165658f61b0d8e3a.png: 256x256 54 nucleuss, 12.3ms
Speed: 0.5ms preprocess, 12.3ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)

image 1/1 /content/yolo_cell_nuclei/images/val/62570c4ff1c5ab6d9d383aba9f25e604768520b4266afd40fdf4734a694c8bc3.png: 256x256 10 nucleuss, 12.5ms
Speed: 0.6ms preprocess, 12.5ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 256)


Dice Computation: 100%|██████████| 134/134 [00:09<00:00, 14.70it/s]

✅ Mean Dice Score on Validation Set: 0.7359
